# Bonus Session · Claude Shannon and Language as Redundant Pattern
*Cultural Machines: An Introduction*
Based on Leif Weatherby, *Language Machines: Cultural AI and the End of Remainder Humanism* (University of Minnesota Press, 2025).

**In this session you will:** play the guessing game Shannon used to measure English, rebuild his famous "approximations to English" from a nineteenth-century novel, measure how predictable (and how free) language is, revisit the Chomsky–Shannon argument with a real language model, and ask whose English Shannon's theory was built on.

**Where it fits:** this works well straight after Session 1, since everything a chatbot does rests on Shannon's idea of predicting the next piece. It also sets up Session 2.

**Time:** about 75–90 minutes. Parts 1–4 and 6–7 run on a normal (CPU) runtime; Part 5 loads GPT-2.

### How to use this notebook
- This is a **Google Colab notebook**: a page that mixes reading with small pieces of code you can run.
- To run a grey code box, click it and press **Shift + Enter** (or click the ▶ button on its left).
- **Run the boxes in order, top to bottom.** If something breaks, go to *Runtime → Restart session* and start again from the top.
- You never have to *write* code. Where you see text inside quotation marks, like `"this"`, you can change the words and run the box again. That is the whole skill.
- Boxes marked **Setup** load the machinery. You can open them if you are curious, but you do not need to read them.

## The big idea

In 1948 Claude Shannon, an engineer at Bell Labs, published "A Mathematical Theory of Communication" and founded **information theory**, the mathematics behind phone networks, compression and the internet. His question was practical: how do you send a message down a noisy line without losing anything?

To answer it, he studied English as a pattern. Some letters and words follow others far more often than chance: after *q* comes *u*; after *in the event* comes *that* far more often than *elephant*. Shannon called this **redundancy**, and estimated that English is roughly half redundant: you can lose about half of it and still recover the message. The other side of redundancy is what he called the **freedom of choice** a speaker has at each step, which he measured as **entropy**, or surprise.

Shannon famously said the *meaning* of messages was irrelevant to the engineering problem, and he is usually remembered as the man who took meaning out of communication. Weatherby argues the opposite. Redundancy, he says, is itself a theory of meaning: meaning comes from the **structure and density of patterns** in language as a whole, not from rules in the head or words pointing at things. In that sense Shannon worked with a **non-referential theory of language**, and LLMs are his idea scaled up beyond anything he imagined.

Weatherby also takes three more things from Shannon, which we test in this notebook:
- **Nothing is ever fully certain or fully impossible.** No next word is guaranteed, and even nonsense can be given a meaning.
- **The part reflects the whole.** Shannon assumed any long enough stretch of text mirrors the patterns of the language (his technical word was *ergodic*). Weatherby says LLMs more or less prove it.
- **Shannon's English was not universal.** Following the scholar Lydia Liu, Weatherby notes that Shannon built his theory on "printed English" and treated it as the model for all communication.

## Setup

In [ ]:
#@title Setup: load a novel and build the tools (about 30 seconds)
import re, math, random, urllib.request
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ALPHABET = "abcdefghijklmnopqrstuvwxyz "   # Shannon's 27 symbols: 26 letters and a space

def clean(text):
    """Lower-case, keep only a–z, turn everything else into single spaces."""
    text = re.sub(r"[^a-z]+", " ", text.lower())
    return re.sub(r" +", " ", text).strip()

def load_gutenberg(url):
    raw = urllib.request.urlopen(url, timeout=30).read().decode("utf-8", errors="ignore")
    start = raw.find("*** START OF")
    if start != -1:
        raw = raw[raw.find("\n", start) + 1:]
    end = raw.find("*** END OF")
    if end != -1:
        raw = raw[:end]
    return raw

FALLBACK = """
The market opened early on the morning of the festival. Traders arrived before the sun,
carrying baskets of tomatoes, onions, peppers and fish, and the drummers began to practise
at the edge of the square. By the time the first visitors came, the whole street was full
of voices, music and the smell of cooking. Children ran between the stalls while their
parents argued over prices. An old woman sold cloth in bright patterns, and a young man
painted portraits for anyone who would sit still long enough. In the afternoon the rain
came, as it always does in that season, and everyone crowded under the roofs of the shops
to wait. When the rain stopped the music started again, louder than before, and the dancing
went on until long after dark. People said later that it was the best festival in years,
though they said the same thing every year, and nobody minded.
""" * 20

BOOK_URL = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"   # Pride and Prejudice (public domain)
try:
    raw_book = load_gutenberg(BOOK_URL)
    print("Loaded Pride and Prejudice from Project Gutenberg.")
except Exception as e:
    raw_book = FALLBACK
    print("Could not download the novel, so using a short built-in text instead. Results will be rougher.")

corpus = clean(raw_book)
words = corpus.split()
print(f"{len(corpus):,} characters and {len(words):,} words to learn from.")

# ---------- letter statistics ----------
def build_counts(text, max_n=6):
    counts = {n: defaultdict(Counter) for n in range(1, max_n + 1)}
    for n in range(1, max_n + 1):
        for i in range(len(text) - n + 1):
            counts[n][text[i:i+n-1]][text[i+n-1]] += 1
    return counts

COUNTS = build_counts(corpus)

def _next_dist(context, order):
    """Distribution of the next letter, backing off to shorter contexts if needed."""
    for n in range(order, 0, -1):
        ctx = context[-(n-1):] if n > 1 else ""
        if len(ctx) == n - 1 and ctx in COUNTS[n] and COUNTS[n][ctx]:
            return COUNTS[n][ctx]
    return COUNTS[1][""]

def approximation(order, length=250, seed=None):
    """Shannon's 'approximations to English'. order 0 = random letters, 1 = letter frequencies, 2+ = letters after letters."""
    rnd = random.Random(seed)
    if order == 0:
        return "".join(rnd.choice(ALPHABET) for _ in range(length))
    out = ""
    while len(out) < length:
        dist = _next_dist(out, order)
        letters, weights = zip(*dist.items())
        out += rnd.choices(letters, weights)[0]
    return out

WORD1 = Counter(words)
WORD2 = defaultdict(Counter)
for a, b in zip(words, words[1:]):
    WORD2[a][b] += 1

def word_approximation(order, length=40, seed=None):
    rnd = random.Random(seed)
    vocab, freq = zip(*WORD1.items())
    out = [rnd.choices(vocab, freq)[0]]
    while len(out) < length:
        if order >= 2 and WORD2[out[-1]]:
            nxt, w = zip(*WORD2[out[-1]].items())
            out.append(rnd.choices(nxt, w)[0])
        else:
            out.append(rnd.choices(vocab, freq)[0])
    return " ".join(out)

def show(text, width=90):
    import textwrap
    print(textwrap.fill(text, width))

def next_letter_table(context, k=8):
    """What letters follow this context, and how often?"""
    ctx = re.sub(r" +", " ", re.sub(r"[^a-z ]+", " ", context.lower()))
    order = min(len(ctx) + 1, 6)
    dist = _next_dist(ctx, order)
    total = sum(dist.values())
    shown = ctx[-(order - 1):] if order > 1 else ""
    rows = [("space" if c == " " else c, f"{v/total*100:.1f}%") for c, v in dist.most_common(k)]
    return pd.DataFrame(rows, columns=[f"after '{shown}'", "how often"])

def next_word_table(context, k=8):
    prev = clean(context).split()[-1]
    dist = WORD2[prev]
    total = sum(dist.values()) or 1
    return pd.DataFrame([(w, f"{v/total*100:.1f}%") for w, v in dist.most_common(k)],
                        columns=[f"after '{prev}'", "how often"])

def entropy(counter):
    total = sum(counter.values())
    return -sum(v/total * math.log2(v/total) for v in counter.values() if v)

def conditional_entropy(n):
    """Average uncertainty (in bits) about the next letter, knowing the previous n-1 letters."""
    table = COUNTS[n]
    grand = sum(sum(c.values()) for c in table.values())
    return sum(sum(c.values()) / grand * entropy(c) for c in table.values())

def delete_letters(sentence, fraction=0.5, seed=1):
    rnd = random.Random(seed)
    return "".join("_" if ch.isalpha() and rnd.random() < fraction else ch for ch in sentence)

def guessing_game(sentence):
    """Shannon's guessing game: guess the hidden sentence one letter at a time."""
    target = clean(sentence)
    print("Guess the hidden sentence one letter at a time.")
    print("Type one letter and press Enter. For a gap between words, just press Enter. Type ? to reveal a letter.\n")
    shown, tries = "", []
    for ch in target:
        n = 0
        while True:
            g = input(f"{shown}▮   your guess: ")
            g = " " if g.strip() == "" else g.strip().lower()[:1]
            n += 1
            if g == "?" or g == ch:
                break
            print("   no")
        tries.append(n); shown += ch
    print("\n" + target)
    print("".join(str(t) if t < 10 else "+" for t in tries))
    first = sum(1 for t in tries if t == 1) / len(tries)
    print(f"\nYou guessed {first*100:.0f}% of the letters on your first try.")
    return tries

def machine_guess(sentence, order=5):
    """The same game, played by the letter statistics of the novel."""
    target = clean(sentence)
    tries = []
    for i, ch in enumerate(target):
        ranking = [c for c, _ in _next_dist(target[:i], order).most_common()]
        ranking += [c for c in ALPHABET if c not in ranking]
        tries.append(ranking.index(ch) + 1)
    print(target)
    print("".join(str(t) if t < 10 else "+" for t in tries))
    first = sum(1 for t in tries if t == 1) / len(tries)
    print(f"\nThe machine guessed {first*100:.0f}% of the letters on its first try.")
    return tries

def letter_freqs(text):
    c = Counter(ch for ch in text if ch in ALPHABET)
    total = sum(c.values()) or 1
    return np.array([c[ch] / total for ch in ALPHABET])

print("Ready.")

## Part 1 · Shannon's guessing game

In 1951 Shannon measured English with a simple game. He picked a sentence and asked a person to guess it one letter at a time, recording how many guesses each letter took. Most letters were guessed on the first try. That is redundancy you can feel.

Play it yourself. Run the box, then type one letter per guess and press Enter. For the gap between words, just press Enter on its own. Type `?` to give up on a letter.

In [ ]:
my_tries = guessing_game("the concert starts at eight")

Under the sentence you'll see how many guesses each letter took. Where were you fastest? Usually in the middle and at the end of words, and in common phrases. Where were you slowest? Usually at the **start** of a word, where your freedom of choice is greatest.

Now let the letter statistics of *Pride and Prejudice* play the same game. It knows nothing about concerts, only which letters tend to follow which.

In [ ]:
machine_tries = machine_guess("the concert starts at eight")

In [ ]:
# Try your own sentences. Which are easy to predict? Which are hard?
machine_guess("the rain in the afternoon stopped the music")
machine_guess("kwame mensah exhibited seven sculptures")

**Notice:** names and unusual words are expensive; function words and word endings are cheap. The machine is doing exactly what you did, only with counted statistics instead of intuition.

## Part 2 · Delete half the letters

If English is about half redundant, you should be able to read it with half the letters gone. Try reading these before revealing the originals.

In [ ]:
sentence = "The festival will open on Friday evening with a concert in the main square."
for share in [0.25, 0.5, 0.7]:
    print(f"{int(share*100)}% of letters removed:  ", delete_letters(sentence, share))

In [ ]:
print(sentence)

Somewhere around half, reading turns from easy to hard. This is what allowed Shannon to calculate how much a message could be compressed, and how much error-protection a noisy channel needed: the idea of **bandwidth** that shapes our entire technical world.

## Part 3 · Approximations to English

In his 1948 paper Shannon built a ladder of fake Englishes, each using a little more of the language's statistics. We rebuild his ladder here from *Pride and Prejudice*. Read each rung aloud.

In [ ]:
labels = {
    0: "Random letters (all 27 symbols equally likely)",
    1: "Letters at their real frequencies",
    2: "Each letter chosen given the one before it",
    3: "Each letter chosen given the two before it",
    4: "Each letter chosen given the three before it",
    5: "Each letter chosen given the four before it",
}
for order, label in labels.items():
    print(f"\n=== Rung {order}: {label} ===")
    show(approximation(order, length=200, seed=3))

Now the same ladder with **words** instead of letters.

In [ ]:
print("=== Words at their real frequencies ===")
show(word_approximation(1, seed=4))
print("\n=== Each word chosen given the word before it ===")
show(word_approximation(2, seed=4))

**Look for:** at which rung does it start to *sound* like English? At which rung does it start to sound like **this particular novel**, with its drawing rooms, sisters and gentlemen? No rule of grammar was given to the machine at any point. Only counting.

This is Weatherby's point in miniature. Style, genre and even a flavour of the novel's world arrive through pattern alone, before anything like meaning in the dictionary sense. A modern LLM is this ladder climbed almost to the top, with far longer contexts than a few letters.

## Part 4 · Measuring freedom

Shannon's key quantity is how much freedom you have at each step. Compare what can follow these contexts in the novel:

In [ ]:
display(next_letter_table("q"))
display(next_letter_table("th"))
display(next_letter_table("the "))

In [ ]:
# Word-level: compare a wide-open context with a narrow one
display(next_word_table("the"))
display(next_word_table("in the event"))

After *q* there is almost no freedom. After *the* there is a lot. Shannon measured this freedom in **bits**: roughly, how many yes/no questions you would need to ask to find the next letter. The chart below shows how the average uncertainty about the next letter falls as the machine is allowed to look further back.

In [ ]:
max_bits = math.log2(27)
rows = [("random letters", max_bits)] + [(f"{n - 1} letters" if n > 1 else "frequencies only", conditional_entropy(n))
                                         for n in range(1, 7)]
df = pd.DataFrame(rows, columns=["what the machine knows", "uncertainty (bits per letter)"])
df["redundancy estimate"] = (1 - df["uncertainty (bits per letter)"] / max_bits).map(lambda r: f"{r*100:.0f}%")
display(df)

plt.figure(figsize=(7, 4))
plt.plot(df["what the machine knows"], df["uncertainty (bits per letter)"], marker="o", color="darkslateblue")
plt.xticks(rotation=30, ha="right")
plt.xlabel("how many previous letters the machine can see")
plt.ylabel("bits per letter"); plt.title("The more context, the less freedom")
plt.tight_layout(); plt.show()

**A caution:** with six letters of context, a single novel starts to run out of examples, so the last points are over-optimistic. Shannon's own estimates, using human guessers and much longer contexts, went lower still, to around one bit per letter, which means that over long stretches English is even more redundant than the famous "about half".

**Weatherby's reading.** The standard story says Shannon measured only this statistical shape and ignored meaning. Weatherby asks: what if meaning *is* this shape? Every message, he suggests, is doubled: it says something, and at the same time it carries a tendency, a pull towards what usually comes next. Meaning sits in the relation between the part (this sentence) and the whole (the language).

## Part 5 · Nothing is impossible: Chomsky versus Shannon

In 1957 the linguist Noam Chomsky attacked Shannon's statistical picture of language with a now-famous sentence: *"Colorless green ideas sleep furiously."* Nobody had ever said it, so its probability should be zero, yet any English speaker recognises it as grammatical, unlike the same words backwards. Chomsky concluded that grammar lives in the mind, not in statistics, and linguistics largely followed him for half a century.

Shannon's collaborator Warren Weaver had made a different point with an even stranger phrase, *"Constantinople fishing nasty pink"*: in real language no sequence ever has zero probability, and anything can be given some meaning. Weatherby sides with Weaver. LLMs, built on next-word probability, reopen the argument Chomsky seemed to have won.

We can test it. GPT-2 measures **surprise** in bits for each piece of a sentence. Lower means more expected.

In [ ]:
#@title Setup: load GPT-2 to measure surprise (about a minute)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
gtok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2"); gpt2.eval()

def surprise(sentence):
    """Surprise, in bits, of each word-piece given what came before it."""
    ids = gtok(gtok.bos_token + sentence, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        logp = torch.log_softmax(gpt2(ids).logits[0, :-1], dim=-1)
    target = ids[0, 1:]
    bits = (-logp[range(len(target)), target] / math.log(2)).tolist()
    pieces = [gtok.decode([int(t)]) for t in target]
    return pieces, bits

def compare_surprise(sentences):
    rows = []
    for s in sentences:
        _, bits = surprise(s)
        rows.append((s, round(sum(bits), 1), round(sum(bits) / len(bits), 1)))
    df = pd.DataFrame(rows, columns=["sentence", "total surprise (bits)", "average per piece"])
    plt.figure(figsize=(9, 0.6 * len(sentences) + 1))
    plt.barh([r[0] for r in rows][::-1], [r[2] for r in rows][::-1], color="darkslateblue")
    plt.xlabel("average surprise per piece (bits): lower = more expected")
    plt.tight_layout(); plt.show()
    return df

def surprise_chart(sentence):
    pieces, bits = surprise(sentence)
    plt.figure(figsize=(0.6 * len(pieces) + 2, 3))
    plt.bar(range(len(pieces)), bits, color="darkslateblue")
    plt.xticks(range(len(pieces)), [p.strip() or "·" for p in pieces], rotation=45, ha="right")
    plt.ylabel("surprise (bits)"); plt.title(sentence); plt.tight_layout(); plt.show()

print("GPT-2 ready.")

In [ ]:
compare_surprise([
    "The children played in the garden until it got dark.",
    "Colorless green ideas sleep furiously.",
    "Furiously sleep ideas green colorless.",
    "Constantinople fishing nasty pink.",
])

**Look for:** does the model find Chomsky's grammatical nonsense *less* surprising than the same words scrambled? If so, a purely statistical system has picked up the difference Chomsky thought it never could. (The computer scientist Fernando Pereira made this argument in 2000 with a far simpler model.) And note that none of these sentences scores as *impossible*: every one gets a finite amount of surprise, just as Weaver said.

Now look inside one sentence, piece by piece:

In [ ]:
surprise_chart("The curator hung the paintings, and the paintings hung the curator.")

In [ ]:
# Try: a proverb, a line of your own poetry, a sentence with a name in it,
# the opening of a funding bid...
surprise_chart("Write your own sentence here.")

Where does surprise spike? Often at names, at a twist in meaning, at the word a poet chose *against* expectation. This is where Weatherby locates the difference between **ideology** (the low-surprise, well-worn path, see Session 5) and **poetry** (language that is improbable yet rich in meaning).

## Part 6 · The part reflects the whole

Shannon assumed that any long enough stretch of text reflects the statistics of the whole language. Let's check. We take random chunks of the novel, of increasing length, and measure how far their letter frequencies are from the whole book's.

In [ ]:
whole = letter_freqs(corpus)
lengths = [30, 100, 300, 1000, 3000, 10000, 30000, 100000]
rnd = random.Random(0)
avg_gap = []
for L in lengths:
    gaps = []
    for _ in range(30):
        start = rnd.randrange(0, max(1, len(corpus) - L))
        chunk = letter_freqs(corpus[start:start + L])
        gaps.append(0.5 * np.abs(chunk - whole).sum() * 100)
    avg_gap.append(np.mean(gaps))

plt.figure(figsize=(7, 4))
plt.plot(lengths, avg_gap, marker="o", color="teal")
plt.xscale("log"); plt.xlabel("length of the chunk (letters, log scale)")
plt.ylabel("difference from the whole book (%)")
plt.title("A long enough part looks like the whole"); plt.tight_layout(); plt.show()

A tweet-length chunk is lopsided; a chapter-length chunk looks almost exactly like the whole. Weatherby says LLMs more or less prove this assumption of Shannon's, because they generate each word by relating the text so far (the part) to everything they learned (the whole).

## Part 7 · Whose English?

Shannon's alphabet had 27 symbols, and his model of all communication was **printed English**. Lydia Liu's research, which Weatherby draws on, shows how much that choice carried with it: one language, one alphabet, and a particular written culture, quietly treated as universal. Weatherby mentions an alternative road: the Cambridge researcher Margaret Masterman argued that a real theory of language for machines should start from Chinese instead.

Test this yourself. Paste a paragraph (a few hundred words is best) in another language you know: Twi, Ewe, Ga, Swahili, French, Hausa... Then run the box. Note what the cleaning step does to it.

In [ ]:
my_text = """Paste a few paragraphs in another language here."""

print("What Shannon's 27-symbol alphabet keeps of your text:\n")
show(clean(my_text)[:400])
lost = sorted(set(ch for ch in my_text.lower() if ch.isalpha() and ch not in ALPHABET))
print(f"\nLetters thrown away because they are not in the English alphabet: {' '.join(lost) or 'none'}")

In [ ]:
yours = letter_freqs(clean(my_text))
x = np.arange(len(ALPHABET))
plt.figure(figsize=(12, 4))
plt.bar(x - 0.2, whole * 100, width=0.4, label="Pride and Prejudice (English)")
plt.bar(x + 0.2, yours * 100, width=0.4, label="your text")
plt.xticks(x, [c if c != " " else "space" for c in ALPHABET])
plt.ylabel("% of symbols"); plt.legend(); plt.title("Different languages, different patterns")
plt.tight_layout(); plt.show()

**Notice:** languages such as Twi or Ewe use letters like ɛ and ɔ, and many use tone marks. Shannon's alphabet simply deletes them, which changes the words. Early encoding systems for computers made similar choices, and some of their consequences are still visible in how AI handles African languages (see Session 5).

## Discussion

1. Shannon said meaning was irrelevant to his engineering problem. After this session, do you agree with Weatherby that redundancy is itself a kind of theory of meaning?
2. Chomsky versus Shannon is often told as "the mind" versus "statistics". Where would you place poets, griots, translators or editors in that argument?
3. Low surprise is the well-worn path; high surprise is either noise or poetry. How do *you* tell the difference in your own work?
4. What would information theory look like if it had been built on Twi, Chinese or Arabic rather than printed English?

## Glossary
- **Information theory**: Shannon's mathematics of communication (1948).
- **Redundancy**: the patterning that makes language predictable, and so compressible and recoverable.
- **Entropy / surprise**: how unpredictable the next symbol is, measured in bits.
- **Bit**: one yes/no question's worth of information.
- **Freedom of choice**: Shannon's phrase for the range of options open at each step of a message.
- **Ergodic**: Shannon's assumption that a long enough part of a text shows the patterns of the whole.
- **Markov chain / n-gram**: choosing each symbol based on the few symbols before it, as in Part 3.

## Going further
- Weatherby, *Language Machines*, chapter 4 (Shannon and redundancy), with shorter appearances in chapters 1, 5 and 6.
- Claude Shannon, "A Mathematical Theory of Communication" (1948), and "Prediction and Entropy of Printed English" (1951), the guessing-game paper.
- James Gleick, *The Information* (2011), a very readable history.
- Lydia H. Liu, *The Freudian Robot* (2010), on Shannon's "printed English".